In [2]:
import sys, platform, os
from pathlib import Path

print("Python      :", sys.version.split()[0])
print("Ejecutable  :", sys.executable)
print("Sistema     :", platform.system(), platform.release())
print("Arquitectura:", platform.machine())
print("Directorio  :", Path.cwd())

entorno = Path(sys.executable).parent.name
print("\nEntorno detectado:", entorno)
if "ec2053c" not in entorno.lower():
    print("AVISO: el kernel no parece ser el entorno ec2053c. Reviselo antes de continuar.")
else:
    print("OK: kernel correcto.")

Python      : 3.12.13
Ejecutable  : C:\Users\Jose\miniconda3\envs\ec2053c\python.exe
Sistema     : Windows 10
Arquitectura: AMD64
Directorio  : C:\Users\Jose

Entorno detectado: ec2053c
OK: kernel correcto.


In [4]:
import importlib

REQUERIDOS = {
    "numpy":        "1.26",
    "pandas":       "2.1",
    "scipy":        "1.11",
    "sklearn":      "1.4",
    "matplotlib":   "3.8",
    "pyarrow":      "14.0",
}
OPCIONALES = {
    "statsmodels":  "0.14",
    "seaborn":      "0.13",
}

def _tupla(v):
    partes = []
    for p in str(v).split("."):
        num = "".join(ch for ch in p if ch.isdigit())
        partes.append(int(num) if num else 0)
    return tuple(partes)

def revisar(paquetes, obligatorio=True):
    filas = []
    for nombre, minimo in paquetes.items():
        try:
            mod = importlib.import_module(nombre)
            version = getattr(mod, "__version__", "desconocida")
            ok = version == "desconocida" or _tupla(version) >= _tupla(minimo)
            estado = "OK" if ok else "VERSION BAJA"
        except ImportError:
            version, estado = "-", ("FALTA" if obligatorio else "opcional, no instalado")
        filas.append((nombre, minimo, version, estado))
    return filas

filas = revisar(REQUERIDOS) + revisar(OPCIONALES, obligatorio=False)

ancho = max(len(f[0]) for f in filas) + 2
print(f"{'paquete':<{ancho}}{'mínima':<10}{'instalada':<14}estado")
print("-" * (ancho + 40))
for nombre, minimo, version, estado in filas:
    print(f"{nombre:<{ancho}}{minimo:<10}{version:<14}{estado}")

faltantes = [f[0] for f in filas[:len(REQUERIDOS)] if f[3] != "OK"]
print()
if faltantes:
    print("Instale o actualice antes de seguir:")
    print("  conda install -n ec2053c " + " ".join(faltantes).replace("sklearn", "scikit-learn"))
else:
    print("OK: todos los paquetes obligatorios están disponibles.")

paquete      mínima    instalada     estado
-----------------------------------------------------
numpy        1.26      2.4.6         OK
pandas       2.1       3.0.3         OK
scipy        1.11      1.18.0        OK
sklearn      1.4       1.9.0         OK
matplotlib   3.8       3.11.0        OK
pyarrow      14.0      23.0.1        OK
statsmodels  0.14      0.14.6        OK
seaborn      0.13      0.13.2        OK

OK: todos los paquetes obligatorios están disponibles.


In [5]:
import numpy as np
import random

SEED = 2053

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

muestra = rng.normal(size=5)
print("Semilla del curso :", SEED)
print("Muestra normal    :", np.round(muestra, 6))
print("Suma de control   :", round(float(muestra.sum()), 10))

ESPERADO = 2.6972271071
coincide = abs(float(muestra.sum()) - ESPERADO) < 1e-9
print("\nValor de referencia:", ESPERADO)
print("OK: reproducible." if coincide else
      "AVISO: la suma no coincide con la referencia. Revise la versión de numpy con su equipo.")

Semilla del curso : 2053
Muestra normal    : [-0.13432   1.251593 -0.449899  2.36988  -0.340027]
Suma de control   : 2.6972271071

Valor de referencia: 2.6972271071
OK: reproducible.


In [6]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CARPETAS = [
    "data/raw",         # extractos originales, nunca se editan
    "data/interim",     # pasos intermedios
    "data/processed",   # matrices de diseño listas para modelar
    "notebooks",        # cuadernos numerados
    "src",              # funciones reutilizables
    "reports",          # informes y figuras
]

for c in CARPETAS:
    (RAIZ / c).mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ, "\n")
for c in CARPETAS:
    ruta = RAIZ / c
    n = len([p for p in ruta.iterdir() if p.is_file()]) if ruta.exists() else 0
    print(f"  {c:<18} {'existe' if ruta.exists() else 'FALTA':<8} {n} archivo(s)")

gitignore = RAIZ / ".gitignore"
if not gitignore.exists():
    gitignore.write_text(
        "data/raw/\ndata/interim/\n.ipynb_checkpoints/\n__pycache__/\n*.pyc\n.env\n",
        encoding="utf-8")
    print("\n.gitignore creado.")
else:
    print("\n.gitignore ya existe.")

print("\nRecordatorio: los datos de la organización NO se suben al repositorio. Sólo el código y la "
      "documentación.")

Raíz del proyecto: C:\Users\Jose 

  data/raw           existe   0 archivo(s)
  data/interim       existe   0 archivo(s)
  data/processed     existe   0 archivo(s)
  notebooks          existe   0 archivo(s)
  src                existe   0 archivo(s)
  reports            existe   0 archivo(s)

.gitignore creado.

Recordatorio: los datos de la organización NO se suben al repositorio. Sólo el código y la documentación.


In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

n = 300
X = pd.DataFrame({
    "monto":    rng.lognormal(mean=10, sigma=0.6, size=n),
    "atraso":   rng.poisson(lam=4, size=n).astype(float),
    "segmento": rng.choice(["Retail", "Mayorista", "Institucional"], size=n),
})
X.loc[rng.choice(n, size=15, replace=False), "atraso"] = np.nan   # faltantes a propósito
y = (X["atraso"].fillna(0) + rng.normal(0, 2, size=n) > 5).astype(int)

num = Pipeline([("imp", SimpleImputer(strategy="median")),
                ("esc", StandardScaler())])
cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                ("oh",  OneHotEncoder(handle_unknown="ignore"))])

pre = ColumnTransformer([("num", num, ["monto", "atraso"]),
                         ("cat", cat, ["segmento"])])

modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
auc = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")

print("AUC por pliegue :", np.round(auc, 4))
print(f"AUC media       : {auc.mean():.4f}  (±{auc.std():.4f})")
print("\nOK: el flujo completo se ejecuta." if auc.mean() > 0.5 else "AVISO: revise la instalación.")

AUC por pliegue : [0.9114 0.8357 0.9281 0.7612 0.8312]
AUC media       : 0.8535  (±0.0604)

OK: el flujo completo se ejecuta.


In [8]:
from datetime import datetime

lineas = [
    "VERIFICACIÓN DE ENTORNO · EC2053C Análisis de Datos II · 2-2026",
    "=" * 66,
    f"Fecha de ejecución : {datetime.now():%Y-%m-%d %H:%M}",
    f"Python             : {sys.version.split()[0]}",
    f"Ejecutable         : {sys.executable}",
    f"Sistema            : {platform.system()} {platform.release()} ({platform.machine()})",
    f"Semilla del curso  : {SEED}",
    f"Reproducibilidad   : {'OK' if coincide else 'REVISAR'}",
    f"Prueba de humo AUC : {auc.mean():.4f}",
    "",
    "Paquetes:",
]
for nombre, minimo, version, estado in filas:
    lineas.append(f"  {nombre:<14} {version:<14} {estado}")

informe = "\n".join(lineas)
destino = RAIZ / "reports" / "entorno_verificado.txt"
destino.write_text(informe, encoding="utf-8")

print(informe)
print("\n" + "-" * 66)
print("Informe guardado en:", destino)

VERIFICACIÓN DE ENTORNO · EC2053C Análisis de Datos II · 2-2026
Fecha de ejecución : 2026-08-15 03:15
Python             : 3.12.13
Ejecutable         : C:\Users\Jose\miniconda3\envs\ec2053c\python.exe
Sistema            : Windows 10 (AMD64)
Semilla del curso  : 2053
Reproducibilidad   : OK
Prueba de humo AUC : 0.8535

Paquetes:
  numpy          2.4.6          OK
  pandas         3.0.3          OK
  scipy          1.18.0         OK
  sklearn        1.9.0          OK
  matplotlib     3.11.0         OK
  pyarrow        23.0.1         OK
  statsmodels    0.14.6         OK
  seaborn        0.13.2         OK

------------------------------------------------------------------
Informe guardado en: C:\Users\Jose\reports\entorno_verificado.txt
